In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, desc


spark = SparkSession.builder.appName("SparkApp").getOrCreate()

customers = spark.read.option("header", True).option("inferschema",True).csv("data/customers.csv")

order_items = spark.read.option("header", True).option("inferschema",True).csv("data/order_items.csv")

orders = spark.read.option("header", True).option("inferschema",True).csv("data/orders.csv")

products = spark.read.option("header", True).option("inferschema",True).csv("data/products.csv")

returns = spark.read.option("header", True).option("inferschema",True).csv("data/returns.csv")

print("Customer Count:", customers.count())
print("Order Items Count:", order_items.count())
print("Orders Count:", orders.count())
print("Product Count:", products.count())
print("Returns Count:", returns.count())

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/16 04:00:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/16 04:00:56 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
                                                                                

Customer Count: 100000


[Stage 13:>                                                         (0 + 2) / 2]

Order Items Count: 3000000


Orders Count: 1000000
Product Count: 50000
Returns Count: 100000


In [3]:
cat_amt = products.groupby("category").sum("unit_cost").show()

+--------------+------------------+
|      category|    sum(unit_cost)|
+--------------+------------------+
|Home & Kitchen| 2901364.330000004|
|        Sports| 2853163.040000003|
|   Electronics|2864604.7399999946|
|      Clothing| 2841424.610000002|
|         Books|2853871.8500000075|
|        Beauty|2919388.7500000037|
|          Toys|2851913.1100000013|
+--------------+------------------+



In [4]:
from pyspark.sql.functions import sum, col, desc

df = orders.join(order_items, "order_id").join(products, "product_id")

top_customers = df.groupBy("customer_id") \
                  .agg(
                      sum(
                          col("quantity") * col("unit_cost")
                      ).alias("total_purchase_amount")
                  ) \
                  .orderBy(desc("total_purchase_amount"))

top_customers.show(10)

[Stage 33:=============================>                            (1 + 1) / 2]

+-----------+---------------------+
|customer_id|total_purchase_amount|
+-----------+---------------------+
|      93094|             122887.1|
|      64560|   116660.08000000002|
|      23289|   109359.41999999998|
|      61218|   107647.11000000003|
|      40442|   104905.09000000001|
|      52275|   103149.83000000003|
|      52034|   102637.66999999998|
|      45631|            102636.03|
|      84830|   102471.69999999998|
|      82593|   101936.83999999997|
+-----------+---------------------+
only showing top 10 rows



In [5]:
result = top_customers.join(customers, "customer_id")

result.select(
    "customer_id",
    "customer_name",
    "total_purchase_amount"
).show(10)

[Stage 44:>                                                         (0 + 2) / 2]

+-----------+--------------+---------------------+
|customer_id| customer_name|total_purchase_amount|
+-----------+--------------+---------------------+
|      63271|Customer_63271|             42974.81|
|      22097|Customer_22097|   31576.069999999996|
|      99817|Customer_99817|             47127.58|
|      73048|Customer_73048|    52697.03000000001|
|      18911|Customer_18911|    50431.00000000001|
|      43935|Customer_43935|   50902.799999999996|
|      69637|Customer_69637|             41915.94|
|      95940|Customer_95940|   50357.350000000006|
|      28664|Customer_28664|             11744.51|
|      73470|Customer_73470|   28258.190000000002|
+-----------+--------------+---------------------+
only showing top 10 rows



In [28]:
from pyspark.sql.functions import year, month,count, sum
orders=orders.withColumn("year", year("order_date"))
orders=orders.withColumn("month", month("order_date"))
orders.groupBy("year", "month").count().orderBy("year", "month").show()

latest_year = orders.agg({"year": "max"}).collect()[0][0]
print(latest_year)

last_year_orders = orders.filter(col("year") == latest_year)

last_year_orders.groupBy("month").count().orderBy("month").show()

+----+-----+-----+
|year|month|count|
+----+-----+-----+
|2024|    1|84682|
|2024|    2|79566|
|2024|    3|84484|
|2024|    4|81545|
|2024|    5|84803|
|2024|    6|82081|
|2024|    7|84637|
|2024|    8|84479|
|2024|    9|82048|
|2024|   10|84360|
|2024|   11|82589|
|2024|   12|84726|
+----+-----+-----+



2024


[Stage 109:>                                                        (0 + 2) / 2]

+-----+-----+
|month|count|
+-----+-----+
|    1|84682|
|    2|79566|
|    3|84484|
|    4|81545|
|    5|84803|
|    6|82081|
|    7|84637|
|    8|84479|
|    9|82048|
|   10|84360|
|   11|82589|
|   12|84726|
+-----+-----+



In [42]:
from pyspark.sql.functions import col

total_orders = order_items.groupBy("product_id").count().withColumnRenamed("count","total_orders")
return_orders=order_items.join(returns,"order_id").groupBy("product_id").count().withColumnRenamed("count","returned_orders")

res=total_orders.join(return_orders, "product_id")

res=res.withColumn("return_percentage",(col("returned_orders")*100)/col("total_orders"))
res.show()  

+----------+------------+---------------+------------------+
|product_id|total_orders|returned_orders| return_percentage|
+----------+------------+---------------+------------------+
|     15790|          58|              6|10.344827586206897|
|      1645|          64|              6|             9.375|
|     19530|          50|              5|              10.0|
|     36224|          62|              6|  9.67741935483871|
|      4101|          56|              8|14.285714285714286|
|     32396|          57|             10| 17.54385964912281|
|      2122|          60|              4| 6.666666666666667|
|     35689|          59|              6|10.169491525423728|
|      1959|          56|              4| 7.142857142857143|
|     39285|          59|              7|11.864406779661017|
|     26087|          67|              4| 5.970149253731344|
|     32414|          74|              6| 8.108108108108109|
|     49308|          78|             11|14.102564102564102|
|     41988|          73

In [44]:
from pyspark.sql.functions import count, desc
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number
df = orders.join(customers, "customer_id")

payment = df.groupBy("state", "payment_mode").count()


window = Window.partitionBy("state").orderBy(desc("count"))

payment = payment.withColumn(
    "rank",
    row_number().over(window)
)


payment.filter(payment.rank == 1).show()           

[Stage 120:>                                                        (0 + 2) / 2]

+-----+----------------+-----+----+
|state|    payment_mode|count|rank|
+-----+----------------+-----+----+
|   CA|             UPI|20246|   1|
|   FL|      Debit Card|20010|   1|
|   GA|     Net Banking|20041|   1|
|   IL|Cash on Delivery|20498|   1|
|   MI|      Debit Card|20416|   1|
|   NC|     Net Banking|19596|   1|
|   NY|      Debit Card|20369|   1|
|   OH|     Net Banking|20351|   1|
|   TX|             UPI|20065|   1|
|   WA|             UPI|20244|   1|
+-----+----------------+-----+----+



In [ ]:
from pyspark.solutions functions import col, countDistinct, sum
